# Laboratory 4: Quantum Finance
## Portfolio Optimization with QAOA

**Objective:** Solve a real-world financial problem: selecting the optimal set of assets to maximize returns while minimizing risk.

**Learning Objectives:**
1. Map a Portfolio Optimization problem to a **QUBO** (Quadratic Unconstrained Binary Optimization) matrix.
2. Convert the QUBO to an **Ising Hamiltonian**.
3. Use **QAOA** to enforce the budget constraint and produce a shortlist of feasible portfolios.
4. Analyze results using the **Efficient Frontier**.

**The Problem:**
Given 4 assets (AAPL, MSFT, JPM, XOM), pick $K=2$ that minimize:
$$\text{Cost} = \lambda \times \text{Risk} - \text{Return} + \text{Penalty}(\text{Budget Constraint})$$

---

> ### Convention note — bit ordering
>
> This lab reports every quantum state in the **lab convention (big-endian)**: in
> $|q_0 q_1 q_2 q_3\rangle$, qubit $0$ is the **leftmost** bit. Asset $i$ maps to qubit $i$,
> so `AAPL MSFT JPM XOM` reads left to right and the optimal portfolio AAPL+MSFT is
> $|1100\rangle$.
>
> **Qiskit uses the opposite (little-endian) order.** A bitstring returned by
> `measure_all()` reads $|q_3 q_2 q_1 q_0\rangle$ — the *same* portfolio would print as
> `0011`. Every bitstring shown to you in this notebook is passed through
> `qiskit_to_lab()` first, so what you read is always big-endian. Watch for this if you
> modify the sampling code: the raw Qiskit counts are **not** in lab order.

---

### Environment Setup

In [ ]:
"""
Environment Setup Module.
Run this cell first to install all required libraries.
"""
!pip install qiskit[visualization] qiskit-aer matplotlib scipy pylatexenc

import math, itertools
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from IPython.display import display

# All randomness in this notebook is seeded, so every run reproduces these numbers.
SEED = 1234


def qiskit_to_lab(bitstring: str) -> str:
    """Convert a Qiskit (little-endian) bitstring to lab (big-endian) convention."""
    return bitstring[::-1]


def lab_to_qiskit(bitstring: str) -> str:
    """Convert a lab (big-endian) bitstring to Qiskit (little-endian) convention."""
    return bitstring[::-1]


def bind(circuit: QuantumCircuit, params: list, values) -> QuantumCircuit:
    """
    Bind parameter *values* to *params* by name.

    Do not pass a bare array to `assign_parameters` or to a primitive: those bind
    positionally, in `circuit.parameters` order, which Qiskit sorts alphabetically
    (beta before gamma). Binding by dict keeps gamma and beta where we put them.
    """
    return circuit.assign_parameters(dict(zip(params, values)))


print("\n ENVIRONMENT SETUP COMPLETE.")

---
## Exercise 1: QUBO Construction and Classical Baseline

**Asset universe:** AAPL, MSFT, JPM, XOM — annualised data 2020–2024.

**The core decision:** we must choose exactly $K=2$ assets from 4. The objective is:
$$\min_{x \in \{0,1\}^4}\; \lambda\, x^T \Sigma\, x - \boldsymbol{\mu}^T x \quad\text{s.t.}\quad \sum_i x_i = K$$

**Why convert to QUBO?** The quantum hardware only knows how to minimise Ising-type Hamiltonians ($\sum Z_iZ_j + \sum Z_i$). The QUBO matrix $Q$ is the intermediate step: it captures the quadratic objective in a form that maps directly to Pauli operators via $x_i = (1-Z_i)/2$.

**Why the penalty term $\lambda_\text{pen}(\sum_i x_i - K)^2$?**
This converts the hard equality constraint $\sum x_i = K$ into a soft penalty added to the objective. Any solution that selects the wrong number of assets incurs a large extra cost, so the quantum circuit learns to avoid infeasible states.

**Getting the penalty coefficient right.** Expanding the penalty (and dropping the constant $K^2$):
$$\lambda_\text{pen}\Big(\sum_i x_i - K\Big)^2 \;=\; \lambda_\text{pen}(1-2K)\sum_i x_i \;+\; 2\lambda_\text{pen}\!\!\sum_{i<j} x_i x_j .$$
The cost is evaluated as $x^TQx = \sum_{i,j} Q_{ij}x_ix_j$, which visits **each unordered pair twice** ($ij$ and $ji$). So the symmetric matrix entry must carry *half* of each $i<j$ coefficient:
$$Q_{ii} = \lambda\Sigma_{ii} - \mu_i + \lambda_\text{pen}(1-2K), \qquad Q_{ij} = \lambda\Sigma_{ij} + \lambda_\text{pen}\;\;(i \neq j).$$
Halving the risk term but *not* the penalty term is a subtle and destructive error: it makes the penalty
$P(m) = \lambda_\text{pen}\big[(1-2K)m + 2m(m-1)\big]$, whose minimum sits at $m \approx (2K{+}1)/4$ rather than at $m=K$. With $K=2$ a **single-asset** portfolio then costs *less* than any feasible pair, and no increase of $\lambda_\text{pen}$ repairs it — the gap $P(1)-P(2) = -\lambda_\text{pen}$ only widens. The cell below asserts the correct behaviour explicitly.

**Why establish a classical brute-force baseline?**
For $n=4$ we can enumerate all $\binom{4}{2} = 6$ portfolios exhaustively in microseconds. This gives us the **ground truth** against which to benchmark QAOA. Without it we would not know if the quantum algorithm is finding the right answer.

**What you will see:** AAPL+MSFT wins at $\lambda=1$ because their high expected returns outweigh their correlation penalty. As $\lambda$ increases the optimum moves to pairs with lower joint risk — first AAPL+JPM, then MSFT+JPM (Exercise 5 traces this sweep).

In [ ]:
"""
QUBO Construction Module.
Builds the portfolio QUBO matrix from historical asset data and
enumerates all feasible portfolios to establish a classical baseline.
"""

# ── STUDENT EXPERIMENT ZONE ────────────────────────────────────────────────
LAM     = 1.0    # Risk-aversion parameter (try 0.5, 2.0, 5.0)
LAM_PEN = 3.0    # Budget penalty strength
K       = 2      # Number of assets to select
# ──────────────────────────────────────────────────────────────────────────

TICKERS = ['AAPL', 'MSFT', 'JPM', 'XOM']
N       = len(TICKERS)

# Annualised expected returns (2020-2024)
MU = np.array([0.340, 0.270, 0.150, 0.220])

# Annualised covariance matrix (2020-2024)
# Computed from daily return series; AAPL-MSFT highly correlated (tech),
# XOM (energy) least correlated with the group but the most volatile on its own.
SIGMA = np.array([
    [0.0841, 0.0588, 0.0313, 0.0203],
    [0.0588, 0.0729, 0.0272, 0.0170],
    [0.0313, 0.0272, 0.0576, 0.0294],
    [0.0203, 0.0170, 0.0294, 0.1225]
])

print("Asset Data Summary")
print("-" * 55)
print(f"{'Ticker':>8}  {'mu (return)':>12}  {'sigma (risk)':>12}  {'Sharpe':>8}")
print("-" * 55)
for i, t in enumerate(TICKERS):
    sigma_i = math.sqrt(SIGMA[i, i])
    print(f"  {t:>6}  {MU[i]*100:>10.1f}%  {sigma_i*100:>10.1f}%  {MU[i]/sigma_i:>8.2f}")

print("\nCorrelation Matrix:")
corr = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        corr[i, j] = SIGMA[i, j] / math.sqrt(SIGMA[i, i] * SIGMA[j, j])
print(f"{'':>8}" + "".join(f"{t:>8}" for t in TICKERS))
for i, t in enumerate(TICKERS):
    print(f"  {t:>6}" + "".join(f"{corr[i,j]:>8.3f}" for j in range(N)))


def build_qubo(lam: float, lam_pen: float, K: int,
               mu: np.ndarray = None, sigma: np.ndarray = None) -> np.ndarray:
    """
    Construct the QUBO matrix Q for portfolio optimization.

    Objective: lam * x^T Sigma x - mu^T x
    Penalty:   lam_pen * (sum_i x_i - K)^2

    Because x^T Q x sums each unordered pair {i,j} twice, the symmetric entry
    Q_ij carries half of the i<j coefficient:

        Q_ii = lam*Sigma_ii - mu_i + lam_pen*(1 - 2K)
        Q_ij = lam*Sigma_ij + lam_pen                  (i != j)

    Using 2*lam_pen off-diagonal (a common slip) double-counts the penalty and
    inverts the constraint: single-asset portfolios become cheaper than feasible
    ones, for every lam_pen > 0.

    Args:
        lam:     Risk-aversion parameter.
        lam_pen: Budget penalty weight.
        K:       Target number of assets.
        mu:      Expected returns (defaults to the 4-asset universe).
        sigma:   Covariance matrix (defaults to the 4-asset universe).

    Returns:
        n x n symmetric QUBO matrix Q.
    """
    mu    = MU    if mu    is None else mu
    sigma = SIGMA if sigma is None else sigma
    n     = len(mu)
    Q     = lam * sigma - np.diag(mu)
    Q    += lam_pen * (1 - 2*K) * np.eye(n) + lam_pen * (np.ones((n, n)) - np.eye(n))
    return Q


def portfolio_cost(x: np.ndarray, Q: np.ndarray) -> float:
    """Evaluate the QUBO objective x^T Q x."""
    return float(x @ Q @ x)


def portfolio_stats(x: np.ndarray, mu: np.ndarray = None, sigma: np.ndarray = None):
    """Expected return and risk (std dev) of an equal-weight portfolio."""
    mu     = MU    if mu    is None else mu
    sigma  = SIGMA if sigma is None else sigma
    n_held = x.sum()
    w      = x / n_held if n_held > 0 else x
    var    = float(w @ sigma @ w)
    return {'return': float(w @ mu), 'risk': math.sqrt(var), 'variance': var}


# ── QUBO matrix ─────────────────────────────────────────────────────────────
Q = build_qubo(LAM, LAM_PEN, K)
print(f"\nQUBO Matrix Q (lambda={LAM}, lambda_pen={LAM_PEN}, K={K}):")
print(f"{'':>8}" + "".join(f"{t:>10}" for t in TICKERS))
for i, t in enumerate(TICKERS):
    print(f"  {t:>6}" + "".join(f"{Q[i,j]:>10.4f}" for j in range(N)))

# ── Sanity check: the penalty must make the FEASIBLE set the global optimum ──
all_states = []
for bits in itertools.product([0, 1], repeat=N):
    x = np.array(bits)
    all_states.append((portfolio_cost(x, Q), ''.join(map(str, bits)), int(x.sum())))
all_states.sort()
_, _, best_popcount = all_states[0]
assert best_popcount == K, (
    f"Penalty is mis-scaled: the global QUBO minimum selects {best_popcount} "
    f"asset(s), not K={K}. Check the off-diagonal coefficient in build_qubo()."
)
print(f"\n[check] global minimum over all 2^{N} = {2**N} states selects exactly "
      f"K={K} assets -> penalty correctly enforces the budget constraint.")

# ── Brute-force enumeration over the feasible set ───────────────────────────
print(f"\nBrute-Force Portfolio Ranking (K={K} assets):")
print("-" * 74)
print(f"{'Rank':>5}  {'Portfolio':>15}  {'Lab ket':>10}  {'Cost':>10}  {'Return':>8}  {'Risk':>8}")
print("-" * 74)

candidates = []
for combo in itertools.combinations(range(N), K):
    x     = np.array([1 if i in combo else 0 for i in range(N)])
    bits  = ''.join(map(str, x))          # already lab (big-endian) order
    cost  = portfolio_cost(x, Q)
    stats = portfolio_stats(x)
    name  = '+'.join(TICKERS[i] for i in combo)
    candidates.append((cost, name, bits, stats))

candidates.sort(key=lambda c: c[0])
for rank, (cost, name, bits, stats) in enumerate(candidates):
    marker = " <- OPTIMAL" if rank == 0 else ""
    print(f"  {rank+1:>3}   {name:>15}  {'|'+bits+'>':>10}  "
          f"{cost:>10.4f}  {stats['return']*100:>6.1f}%  {stats['risk']*100:>6.1f}%{marker}")

optimal_cost, optimal_name, optimal_bits, _ = candidates[0]

# ── Efficient frontier plot ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#e53935', '#1565c0', '#f9ab00', '#6a1b9a', '#00838f', '#558b2f']

for rank, (cost, name, bits, stats) in enumerate(candidates):
    axes[0].scatter(stats['risk']*100, stats['return']*100, c=colors[rank],
                    s=200 if rank == 0 else 80, marker='*' if rank == 0 else 'o',
                    zorder=5, label=f"{name} (rank {rank+1})")
    axes[0].annotate(name, (stats['risk']*100 + 0.2, stats['return']*100), fontsize=8)

axes[0].set_xlabel('Portfolio Risk sigma (%)')
axes[0].set_ylabel('Expected Return mu (%)')
axes[0].set_title('Efficient Frontier - 2-Asset Portfolios', fontsize=12)
axes[0].legend(fontsize=7, loc='lower right')
axes[0].grid(alpha=0.3)

names_sorted = [c[1] for c in candidates]
costs_sorted = [c[0] for c in candidates]
bars = axes[1].barh(names_sorted[::-1], costs_sorted[::-1],
                    color=colors[:len(candidates)][::-1], edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_xlabel('QUBO Cost $x^T Q x$')
axes[1].set_title(f'Portfolio Costs (lambda={LAM})', fontsize=12)
axes[1].grid(alpha=0.3, axis='x')

plt.suptitle(f'Portfolio Optimization - {" | ".join(TICKERS)}', fontsize=13)
plt.tight_layout()
plt.show()
print(f"\nClassical optimal: {optimal_name}  |{optimal_bits}>  (cost = {optimal_cost:.4f})")

---
## Exercise 2: QAOA Portfolio Optimization

We now map the QUBO to a quantum Hamiltonian and run QAOA. The substitution $x_i = (1-Z_i)/2$ converts $x^T Q x$ into:

$$H_\text{portfolio} = \sum_{i<j} \frac{Q_{ij}}{2} Z_i Z_j + \sum_i h_i Z_i + \text{const.}$$

where $h_i = -\frac{1}{2}\sum_j Q_{ij}$.

**Why is this the same circuit as Lab 4?**
The QAOA circuit implements $e^{-i\gamma H_C}$ (Oracle) followed by $e^{-i\beta H_B}$ (Mixer). The Mixer $H_B = \sum_i X_i$ is always the same. The Oracle decomposes into:
- One CNOT–$R_z(2\gamma Q_{ij}/2)$–CNOT per $Z_iZ_j$ term
- One $R_z(2\gamma h_i)$ per $Z_i$ term

The **only** difference from MaxCut is that the $R_z$ angles are now proportional to the financial coefficients $Q_{ij}$ instead of $\pm 1$.

**Why use COBYLA?**
COBYLA is gradient-free — it does not require the parameter-shift rule. This is useful for circuits with many parameters ($2p$ in total) where each PSR gradient evaluation costs $2 \times 2p$ circuit runs. COBYLA needs fewer total evaluations for small $p$. For large $p$ or noisy hardware, gradient-based methods (SPSA, Adam with PSR) become competitive.

**What to observe — and what *not* to expect.**
The penalty term does its job: at $p=2$ essentially **100% of the measured shots are feasible** (exactly 2 assets), up from ~74% at $p=1$. Infeasible states are suppressed, as designed.

But QAOA does **not** single out the winner here, and it is worth understanding why. The six feasible portfolios span a QUBO-cost range of only $0.204$, while the penalty scale is $\lambda_\text{pen}=3$. The energy gap between the best and second-best feasible portfolio is $0.023$. At $p=2$ the optimizer reaches $\langle H\rangle \approx -3.030$, which — to four decimal places — is the **mean energy of the feasible subspace** ($-3.0307$). In other words the circuit prepares something very close to the *uniform superposition over all six feasible portfolios*. The measured probabilities come out near $1/6 \approx 16.7\%$ each, and which one happens to top the histogram changes with the random seed.

So read the output this way: **QAOA narrows 16 candidates down to the 6 feasible ones.** Ranking *within* that shortlist comes from the exact QUBO cost, which Exercise 3 uses. That division of labour — quantum for constraint satisfaction, classical for the final ranking — is a realistic picture of how these hybrid algorithms are used.

**Student Experiments:**
1. Run with `P = 1`. What fraction of shots is feasible? Compare with the classical brute force.
2. Change to `P = 2`. Does the feasible fraction rise? Does the *optimal* state pull clearly ahead of the other five?
3. Change `SEED`. Which quantities are stable, and which move? (Look at $\langle H\rangle$, the feasible fraction, and the rank of the optimum.)
4. Go back to Exercise 1 and set `LAM = 5.0`, then re-run both exercises. Does the classical optimum change? Does QAOA follow the shift?

In [ ]:
"""
QAOA Portfolio Optimization Module.
Maps the QUBO to an Ising Hamiltonian and runs QAOA with p layers.
Compares the quantum result against the brute-force classical baseline.
"""

# ── STUDENT EXPERIMENT ZONE ────────────────────────────────────────────────
P          = 2      # Number of QAOA layers (try 1)
N_SHOTS    = 4000   # Measurement shots for sampling
N_RESTARTS = 5      # COBYLA restarts from different random initial points
# ──────────────────────────────────────────────────────────────────────────


def build_portfolio_hamiltonian(Q: np.ndarray) -> SparsePauliOp:
    """
    Convert a QUBO matrix Q to the Ising Hamiltonian H_portfolio.

    Mapping: x_i = (1 - Z_i)/2, so (dropping an additive constant)
        x^T Q x = sum_{i<j} (Q_ij/2) Z_i Z_j + sum_i h_i Z_i
    with h_i = -(1/2) * sum_j Q_ij.

    Qiskit stores Pauli strings little-endian (rightmost char = qubit 0), so the
    per-term string is reversed before construction. Lab qubit i is Qiskit qubit i.

    Args:
        Q: n x n symmetric QUBO matrix.

    Returns:
        SparsePauliOp for H_portfolio, up to an additive constant.
    """
    n, terms = Q.shape[0], []
    for i in range(n):                                  # ZZ terms
        for j in range(i + 1, n):
            coeff = Q[i, j] / 2
            if abs(coeff) < 1e-12:
                continue
            s = list('I' * n); s[i] = 'Z'; s[j] = 'Z'; s.reverse()
            terms.append((''.join(s), coeff))
    for i in range(n):                                  # Z terms
        h_i = -0.5 * Q[i, :].sum()
        if abs(h_i) < 1e-12:
            continue
        s = list('I' * n); s[i] = 'Z'; s.reverse()
        terms.append((''.join(s), h_i))
    return SparsePauliOp.from_list(terms).simplify()


def ising_constant(Q: np.ndarray) -> float:
    """
    Additive constant dropped by build_portfolio_hamiltonian, so that
        x^T Q x = <H_portfolio> + ising_constant(Q).
    Lets us compare a measured <H> directly against QUBO costs.
    """
    n = Q.shape[0]
    return 0.5 * np.trace(Q) + 0.25 * (Q.sum() - np.trace(Q))


def extract_zz_terms(H: SparsePauliOp) -> list:
    """Extract (qubit_i, qubit_j, coefficient) for every ZZ term of H."""
    n, terms = H.num_qubits, []
    for pauli, coeff in zip(H.paulis, H.coeffs):
        z_idx = [i for i in range(n) if str(pauli)[n - 1 - i] == 'Z']
        if len(z_idx) == 2:
            terms.append((z_idx[0], z_idx[1], float(coeff.real)))
    return terms


def build_qaoa_circuit(n: int, edges_zz: list, p: int) -> tuple:
    """
    Construct a QAOA circuit for a general Ising Hamiltonian.

    Returns:
        (circuit, parameter_list) with parameter_list ordered [gamma..., beta...].
        Always bind through bind() -- never positionally (see the setup cell).
    """
    g  = ParameterVector('γ', p)
    b  = ParameterVector('β', p)
    qc = QuantumCircuit(n, name=f'Portfolio QAOA p={p}')
    qc.h(range(n))
    qc.barrier()
    for k in range(p):
        for (i, j, coeff) in edges_zz:                  # e^{-i gamma coeff Zi Zj}
            qc.cx(i, j)
            qc.rz(2 * g[k] * coeff, j)
            qc.cx(i, j)
        qc.barrier()
        qc.rx(2 * b[k], range(n))                       # mixer e^{-i beta sum Xi}
        qc.barrier()
    return qc, list(g) + list(b)


# ── Build Hamiltonian and circuit ──────────────────────────────────────────
H_port   = build_portfolio_hamiltonian(Q)
zz_terms = extract_zz_terms(H_port)

print(f"Portfolio Hamiltonian ({len(H_port)} Pauli terms):")
for pauli, coeff in zip(H_port.paulis, H_port.coeffs):
    if abs(coeff) > 0.01:
        print(f"  {str(pauli):>8}  {coeff.real:+.4f}")

qaoa_circ, params_list = build_qaoa_circuit(N, zz_terms, P)
display(qaoa_circ.draw('mpl', fold=40))
print(f"\nQAOA circuit: {N} qubits, p={P}, {len(params_list)} parameters")
print(f"Gate count: {qaoa_circ.count_ops()}")

# ── Classical optimisation of <H_portfolio> ────────────────────────────────
estimator = EstimatorV2(options={"backend_options": {"seed_simulator": SEED}})
cost_hist = []


def cost_function(params):
    """<H_portfolio> for the QAOA state at the given (gamma, beta)."""
    bound = bind(qaoa_circ, params_list, params)
    val   = float(estimator.run([(bound, H_port)]).result()[0].data.evs)
    cost_hist.append(val)
    return val


rng = np.random.default_rng(SEED)
best_res, best_hist = None, None
for trial in range(N_RESTARTS):
    cost_hist = []
    x0  = rng.uniform(0, math.pi, len(params_list))
    res = minimize(cost_function, x0, method='COBYLA',
                   options={'maxiter': 400 * P, 'rhobeg': 0.5})
    if best_res is None or res.fun < best_res.fun:
        best_res, best_hist = res, cost_hist.copy()
    print(f"  Trial {trial+1}: E_min = {res.fun:.4f}  evaluations = {len(cost_hist)}")

print(f"\nBest <H_portfolio> = {best_res.fun:.4f}")
print(f"Optimal parameters: gamma={best_res.x[:P].round(3)}, beta={best_res.x[P:].round(3)}")

# ── Sample the optimal circuit ─────────────────────────────────────────────
meas_circ = bind(qaoa_circ, params_list, best_res.x)
meas_circ.measure_all()
counts_qiskit = AerSimulator(seed_simulator=SEED).run(meas_circ, shots=N_SHOTS).result().get_counts()

# Convert EVERY bitstring to lab (big-endian) convention before display.
counts = {qiskit_to_lab(s): c for s, c in counts_qiskit.items()}

print(f"\nTop measurement results ({N_SHOTS} shots, lab convention |q0 q1 q2 q3>):")
print("-" * 72)
for state, cnt in sorted(counts.items(), key=lambda kv: -kv[1])[:8]:
    x_arr = np.array([int(b) for b in state])
    if x_arr.sum() == K:
        name = '+'.join(TICKERS[i] for i in range(N) if x_arr[i] == 1)
        tag, bf_cost = f"[{name}]", portfolio_cost(x_arr, Q)
        cost_s = f"cost={bf_cost:.4f}"
    else:
        tag, cost_s = '[infeasible]', 'cost=      -'
    mark = " <- classical optimum" if state == optimal_bits else ''
    print(f"  |{state}>  {cnt:5d} shots  ({cnt/N_SHOTS*100:5.1f}%)  {tag:>20}  {cost_s}{mark}")

p_feasible = sum(c for s, c in counts.items()
                 if sum(int(b) for b in s) == K) / N_SHOTS
print(f"\nFeasible fraction (exactly K={K} assets): {p_feasible*100:.1f}%")
print(f"P(classical optimum |{optimal_bits}>)        : "
      f"{counts.get(optimal_bits, 0)/N_SHOTS*100:.1f}%   "
      f"(uniform over {len(candidates)} feasible = {100/len(candidates):.1f}%)")

# ── What state did QAOA actually prepare? ──────────────────────────────────
# Map QUBO costs onto the H scale so <H> can be compared directly.
ISING_CONST     = ising_constant(Q)
H_ground        = optimal_cost - ISING_CONST                     # best feasible == global min
H_feasible_mean = np.mean([c - ISING_CONST for c, *_ in candidates])

print(f"\nWhich state did QAOA prepare?")
print(f"  ground-state energy   H_min          = {H_ground:.4f}")
print(f"  mean over the {len(candidates)} feasible states     = {H_feasible_mean:.4f}")
print(f"  measured <H_portfolio>               = {best_res.fun:.4f}")
print(f"  |<H> - feasible mean|                = {abs(best_res.fun - H_feasible_mean):.4f}")
print("  -> <H> sits on the feasible-subspace MEAN, not on the ground state:")
print("     the circuit prepares ~the uniform superposition over feasible portfolios.")

display(plot_histogram(counts, title=f"QAOA Portfolio, p={P} (lab convention)",
                       figsize=(10, 4)))

---
## Exercise 3: Business Analysis and Result Interpretation

The QAOA output is a **probability distribution**, not a single answer. Exercise 2 showed what that distribution actually contains: almost all of its weight sits on the **feasible** portfolios, spread nearly evenly across them.

**How to read the output:**
- The feasible fraction tells you whether the penalty is doing its job. Near 100% means the constraint is satisfied; a large infeasible tail means you should raise $\lambda_\text{pen}$ or $p$.
- The set of feasible bitstrings that appear is the **shortlist**. For $n=4$ it is all six of them; for larger $n$ the quantum step genuinely prunes the space.
- Within the shortlist, rank by **exact QUBO cost**, not by shot count. At this problem size the cost differences between feasible portfolios ($\sim 0.02$) are far below the resolution QAOA achieves at small $p$, so the shot ordering is dominated by sampling noise.

**Why is the second-best portfolio useful in practice?**
Financial models have parameter uncertainty — the covariance matrix $\Sigma$ and expected returns $\mu$ are estimates from historical data. A portfolio that is optimal under the model may underperform if the estimates are off. The second-best portfolio often has a different risk profile and serves as a hedge against model error. A classical solver gives you one answer; QAOA gives you a ranked shortlist and a measure of how tightly the candidates are packed.

**What this cell computes:**
1. The top-3 feasible portfolios, **ranked by exact QUBO cost**, each annotated with the probability QAOA assigned it.
2. Their expected return, risk, and Sharpe ratio (computed from the asset data, not the QUBO).
3. The sub-optimality gap: how much worse (in QUBO cost) the second and third recommendations are compared to the optimum.

In [ ]:
"""
Business Analysis Module.
Ranks the feasible portfolios by exact QUBO cost and reports the
probability QAOA assigned to each, plus their financial characteristics.
"""

# ── Collect every feasible portfolio seen in the QAOA sample ───────────────
feasible_results = []
for state, cnt in counts.items():
    x_arr = np.array([int(b) for b in state])           # already lab order
    if x_arr.sum() != K:
        continue
    feasible_results.append({
        'name':   '+'.join(TICKERS[i] for i in range(N) if x_arr[i] == 1),
        'state':  state,
        'prob':   cnt / N_SHOTS,
        'cost':   portfolio_cost(x_arr, Q),
        **portfolio_stats(x_arr),
    })

# Rank by EXACT COST (see Exercise 3 notes): shot counts are near-degenerate here.
by_cost = sorted(feasible_results, key=lambda r: r['cost'])
top3    = by_cost[:3]

print("Business Analysis: Top-3 Portfolios (ranked by exact QUBO cost)")
print("=" * 70)
for rank, r in enumerate(top3):
    is_opt = r['name'] == optimal_name
    print(f"\n  Rank {rank+1}: {r['name']}  |{r['state']}>"
          f"{'   <- classical optimum' if is_opt else ''}")
    print(f"    QAOA probability:    {r['prob']*100:.1f}%")
    print(f"    Expected return:     {r['return']*100:.1f}% p.a.")
    print(f"    Portfolio risk:      {r['risk']*100:.1f}% p.a.")
    print(f"    Sharpe (approx):     {r['return']/r['risk']:.2f}")
    print(f"    QUBO cost:           {r['cost']:.4f}")
    if not is_opt:
        gap = abs(r['cost'] - optimal_cost)
        print(f"    Sub-optimality gap:  {gap:.4f} ({gap/abs(optimal_cost)*100:.2f}% of optimal)")

probs = [r['prob'] for r in feasible_results]
print(f"\n  Shortlist size: {len(feasible_results)} feasible portfolios")
print(f"  QAOA probability spread across them: "
      f"{min(probs)*100:.1f}% - {max(probs)*100:.1f}% "
      f"(uniform would be {100/len(feasible_results):.1f}%)")
print("  -> the quantum step supplies the shortlist; the cost column ranks it.")

# ── Comparison figure ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#2e7d32', '#1565c0', '#f9ab00', '#9c27b0', '#e53935', '#546e7a']
by_name = {r['name']: r for r in feasible_results}

for idx, (cost, name, bits, stats) in enumerate(candidates):
    prob   = by_name.get(name, {}).get('prob', 0.0)
    is_opt = name == optimal_name
    axes[0].scatter(stats['risk']*100, stats['return']*100,
                    s=max(prob * 3000, 30), c=colors[idx],
                    edgecolors='gold' if is_opt else 'white',
                    linewidths=3 if is_opt else 1, zorder=5, alpha=0.85)
    axes[0].annotate(f"{name}\n({prob*100:.0f}%)",
                     (stats['risk']*100 + 0.2, stats['return']*100), fontsize=7.5)

axes[0].set_xlabel('Portfolio Risk sigma (%)')
axes[0].set_ylabel('Expected Return (%)')
axes[0].set_title(f'Efficient Frontier\n(bubble size ~ QAOA probability, p={P})', fontsize=11)
axes[0].grid(alpha=0.3)
axes[0].legend(handles=[mpatches.Patch(edgecolor='gold', facecolor='white',
                                       lw=2, label='Classical optimum')], fontsize=8)

axes[1].plot(best_hist, 'b-', lw=1.5, alpha=0.85, label=f'QAOA p={P}')
axes[1].axhline(optimal_cost - ISING_CONST, color='green', linestyle='--', lw=2,
                label=f'ground state ({optimal_cost - ISING_CONST:.3f})')
axes[1].axhline(np.mean([c - ISING_CONST for c, *_ in candidates]),
                color='grey', linestyle=':', lw=1.5, label='feasible-subspace mean')
axes[1].set_xlabel('COBYLA evaluation')
axes[1].set_ylabel(r'$\langle H_\mathrm{portfolio}\rangle$')
axes[1].set_title(f'COBYLA Convergence - p={P}', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle(f'QAOA Portfolio Optimization - {" | ".join(TICKERS)}', fontsize=13)
plt.tight_layout()
plt.show()

print("\n-- Summary --")
print(f"QAOA feasible fraction : {p_feasible*100:.1f}%")
print(f"Cost-ranked winner     : {top3[0]['name']}  |{top3[0]['state']}>  (cost {top3[0]['cost']:.4f})")
print(f"Classical optimum      : {optimal_name}  |{optimal_bits}>  (cost {optimal_cost:.4f})")
print(f"Match                  : {'YES' if top3[0]['name'] == optimal_name else 'NO'}")

---
## OPTIONAL Exercise 4: Scaling to n = 6 Assets

We add GOOGL and AMZN to the universe. With $n=6$, $K=3$, there are
$\binom{6}{3} = 20$ possible portfolios — still brute-forceable classically but approaching
the regime where classical enumeration begins to slow.

**Why does this exercise matter?**
The $n=4$ case is pedagogically clean but not practically interesting — any laptop solves it instantly. The $n=6$ case begins to show the circuit scaling properties of QAOA:
- The number of ZZ terms grows as $\binom{n}{2}$: from 6 ($n=4$) to 15 ($n=6$).
- The classical brute-force grows from $\binom{4}{2}=6$ to $\binom{6}{3}=20$ — manageable.
- At $n=50$ the brute-force reaches $\sim 10^{14}$ — intractable. QAOA's $\binom{50}{2}=1225$ ZZ terms per layer is still manageable.

**What to observe.** The same division of labour, only less sharp. At $p=1$ roughly two-thirds of the shots are feasible — the penalty is working, but less decisively than the $\sim$100% we reached at $p=2$ for $n=4$. Within the feasible set the 20 candidates again come out close to uniform ($1/20 = 5\%$ each), so the exact cost column, not the histogram, identifies the winner. This is the expected trend: more states to discriminate with the same number of variational parameters.

In [ ]:
"""
Scaling Module (Optional).
Extends the portfolio universe to n=6 assets (K=3) and compares
circuit complexity and QAOA behaviour against the n=4 case.
"""

TICKERS_6 = ['AAPL', 'MSFT', 'JPM', 'XOM', 'GOOGL', 'AMZN']
N6, K6    = 6, 3
P6        = 1        # QAOA layers for the n=6 run

MU_6 = np.array([0.340, 0.270, 0.150, 0.220, 0.260, 0.300])

# Extended covariance (approximate, 2020-2024)
SIGMA_6 = np.array([
    [0.0841, 0.0588, 0.0313, 0.0203, 0.0620, 0.0550],
    [0.0588, 0.0729, 0.0272, 0.0170, 0.0580, 0.0510],
    [0.0313, 0.0272, 0.0576, 0.0294, 0.0260, 0.0240],
    [0.0203, 0.0170, 0.0294, 0.1225, 0.0180, 0.0190],
    [0.0620, 0.0580, 0.0260, 0.0180, 0.0784, 0.0650],
    [0.0550, 0.0510, 0.0240, 0.0190, 0.0650, 0.0900]
])

# Same corrected builder as Exercise 1 -- the penalty scaling is identical.
Q6  = build_qubo(1.0, 5.0, K6, mu=MU_6, sigma=SIGMA_6)
H6  = build_portfolio_hamiltonian(Q6)
zz6 = extract_zz_terms(H6)

qaoa6, params6 = build_qaoa_circuit(N6, zz6, P6)

print("Scaling Comparison: n=4 vs n=6")
print("-" * 52)
ops4, ops6 = qaoa_circ.count_ops(), qaoa6.count_ops()
print(f"  n=4 (p={P}): {ops4.get('cx',0):>4} CX, {ops4.get('rz',0):>3} Rz, {ops4.get('rx',0):>2} Rx")
print(f"  n=6 (p={P6}): {ops6.get('cx',0):>4} CX, {ops6.get('rz',0):>3} Rz, {ops6.get('rx',0):>2} Rx")
print(f"  ZZ terms: C(4,2)={math.comb(4,2)} -> C(6,2)={math.comb(6,2)} "
      f"(x{math.comb(6,2)/math.comb(4,2):.1f})")
print(f"  Feasible portfolios: C(4,2)={math.comb(4,2)} -> C(6,3)={math.comb(6,3)}")

# ── Brute force over all 2^6 states, then over the feasible set ────────────
all6 = []
for bits in itertools.product([0, 1], repeat=N6):
    x = np.array(bits)
    all6.append((portfolio_cost(x, Q6), ''.join(map(str, bits)), int(x.sum())))
all6.sort()
assert all6[0][2] == K6, "Penalty mis-scaled for n=6: global minimum is infeasible."
print(f"\n[check] global minimum over 2^{N6} = {2**N6} states selects exactly K={K6} assets.")

cands6 = [(c, b) for c, b, m in all6 if m == K6]
print(f"\nBrute-force top 5 of C(6,3)={math.comb(6,3)} feasible portfolios:")
for rank, (cost, bits) in enumerate(cands6[:5]):
    name = '+'.join(TICKERS_6[i] for i, v in enumerate(bits) if v == '1')
    print(f"  {rank+1}. |{bits}>  {name:<24}  cost={cost:.4f}"
          f"{'  <- OPTIMAL' if rank == 0 else ''}")
optimal_bits6, optimal_name6 = cands6[0][1], '+'.join(
    TICKERS_6[i] for i, v in enumerate(cands6[0][1]) if v == '1')

# ── QAOA on n=6 ────────────────────────────────────────────────────────────
estimator6 = EstimatorV2(options={"backend_options": {"seed_simulator": SEED}})


def cost6(params):
    bound = bind(qaoa6, params6, params)
    return float(estimator6.run([(bound, H6)]).result()[0].data.evs)


rng6 = np.random.default_rng(SEED)
res6 = minimize(cost6, rng6.uniform(0, math.pi, len(params6)),
                method='COBYLA', options={'maxiter': 500})

opt6 = bind(qaoa6, params6, res6.x)
opt6.measure_all()
cnt6_qiskit = AerSimulator(seed_simulator=SEED).run(opt6, shots=N_SHOTS).result().get_counts()
cnt6 = {qiskit_to_lab(s): c for s, c in cnt6_qiskit.items()}

p_feas6 = sum(c for s, c in cnt6.items() if sum(int(b) for b in s) == K6) / N_SHOTS
print(f"\nQAOA n=6 result (p={P6}):  <H> = {res6.fun:.4f}")
print(f"  Feasible fraction: {p_feas6*100:.1f}%   "
      f"(uniform over C(6,3)={math.comb(6,3)} feasible = {100/math.comb(6,3):.1f}% each)")
print(f"  Top measured states (lab convention):")
for state, cnt in sorted(cnt6.items(), key=lambda kv: -kv[1])[:5]:
    x = np.array([int(b) for b in state])
    feas = x.sum() == K6
    name = '+'.join(TICKERS_6[i] for i in range(N6) if x[i] == 1) if feas else 'infeasible'
    mark = " <- classical optimum" if state == optimal_bits6 else ""
    print(f"    |{state}>  {cnt/N_SHOTS*100:5.1f}%  {name}{mark}")
print(f"\n  Cost-ranked winner: {optimal_name6}  |{optimal_bits6}>")
print("  As at n=4, the histogram supplies the feasible shortlist; cost ranks it.")

---
## OPTIONAL Exercise 5: Sensitivity Analysis — Risk Aversion $\lambda$

This exercise sweeps $\lambda \in \{0.5, 1.0, 1.5, 2.0, 3.0, 5.0\}$ and identifies which portfolio is optimal at each value.

**Why does $\lambda$ shift the optimal portfolio?**
The QUBO objective balances two terms:
- **Risk term** $\lambda x^T\Sigma x$: penalises portfolios with high return variance. Weight $\lambda$.
- **Return term** $-\mu^T x$: rewards portfolios with high expected return. Weight 1.

At small $\lambda$ (aggressive investor), return dominates: AAPL+MSFT win because they have the two highest $\mu$ values, even though they are highly correlated (tech sector).

As $\lambda$ grows, variance dominates and the optimum moves to **AAPL+JPM**, then to **MSFT+JPM**. The pattern is driven by JPM, which has both the lowest individual variance ($\Sigma_{\text{JPM,JPM}} = 0.0576$) and modest correlations with the tech names.

**A caution about reading correlations.** It is tempting to predict that XOM should win at high $\lambda$, because its covariances with the others are the smallest in the matrix ($\Sigma_{\text{MSFT,XOM}} = 0.0170$ is the smallest off-diagonal entry of all, followed by $\Sigma_{\text{AAPL,XOM}} = 0.0203$). It never does. The reason is that portfolio variance $w^T\Sigma w$ contains the **diagonal** too, and XOM's own variance is by far the largest ($0.1225$, i.e. $\sigma = 35\%$). Low correlation cannot rescue an asset that is individually too volatile. Run the sweep and check this against the numbers rather than trusting the intuition.

**Business interpretation:** This exercise traces the **efficient frontier** — the set of recommended portfolios for different investor risk preferences. In a real application you would run this sweep for a client, show them the frontier, and let them choose their $\lambda$ based on their risk tolerance.

In [ ]:
"""
Sensitivity Analysis Module (Optional).
Computes the optimal portfolio for several values of lambda and plots
the resulting efficient frontier.
"""

lambdas = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
frontier_points = []

print("Lambda Sensitivity Analysis - Classical Brute-Force")
print("-" * 66)
print(f"{'lambda':>7}  {'Optimal Portfolio':>18}  {'Lab ket':>9}  {'Return':>8}  {'Risk':>8}")
print("-" * 66)

prev_opt = None
for lam in lambdas:
    Q_lam = build_qubo(lam, LAM_PEN, K)
    cands = []
    for combo in itertools.combinations(range(N), K):
        x = np.array([1 if i in combo else 0 for i in range(N)])
        cands.append((portfolio_cost(x, Q_lam),
                      '+'.join(TICKERS[i] for i in combo),
                      ''.join(map(str, x)),
                      portfolio_stats(x)))
    cands.sort(key=lambda c: c[0])
    opt_cost, opt_name, opt_bits, opt_stats = cands[0]
    changed = '  <- changed' if prev_opt is not None and opt_name != prev_opt else ''
    print(f"  {lam:>5.1f}  {opt_name:>18}  {'|'+opt_bits+'>':>9}  "
          f"{opt_stats['return']*100:>6.1f}%  {opt_stats['risk']*100:>6.1f}%{changed}")
    frontier_points.append({'lam': lam, 'name': opt_name,
                            'return': opt_stats['return'], 'risk': opt_stats['risk']})
    prev_opt = opt_name

# Check the claim in the markdown: XOM never enters the optimal portfolio.
winners = {p['name'] for p in frontier_points}
print(f"\n  Portfolios that are optimal for some lambda: {sorted(winners)}")
print(f"  XOM appears in an optimal portfolio: {any('XOM' in w for w in winners)}")
print(f"  XOM individual variance = {SIGMA[3,3]:.4f} (largest); "
      f"smallest off-diagonal = {min(SIGMA[i,j] for i in range(N) for j in range(N) if i<j):.4f}")

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
unique_names = list(dict.fromkeys(p['name'] for p in frontier_points))
colors_map   = dict(zip(unique_names, ['#e53935', '#1565c0', '#f9ab00', '#6a1b9a', '#00838f']))

for p in frontier_points:
    axes[0].scatter(p['lam'], p['return']*100, c=colors_map[p['name']], s=120, zorder=5)
axes[0].set_xlabel('Risk Aversion lambda')
axes[0].set_ylabel('Optimal Portfolio Return (%)')
axes[0].set_title('Optimal Return vs Risk Aversion', fontsize=11)
legend_patches = [mpatches.Patch(color=c, label=n) for n, c in colors_map.items()]
axes[0].legend(handles=legend_patches, fontsize=8)
axes[0].grid(alpha=0.3)

all_stats = []
for combo in itertools.combinations(range(N), K):
    x = np.array([1 if i in combo else 0 for i in range(N)])
    all_stats.append({'name': '+'.join(TICKERS[i] for i in combo), **portfolio_stats(x)})

for s in all_stats:
    axes[1].scatter(s['risk']*100, s['return']*100, c='#90a4ae', s=60, zorder=3, alpha=0.7)
    axes[1].annotate(s['name'], (s['risk']*100 + 0.15, s['return']*100),
                     fontsize=7, color='#607d8b')
for fp in frontier_points:
    s = next(s for s in all_stats if s['name'] == fp['name'])
    axes[1].scatter(s['risk']*100, s['return']*100, c=colors_map[fp['name']],
                    s=140, zorder=5, edgecolors='black', linewidths=1)

axes[1].set_xlabel('Portfolio Risk sigma (%)')
axes[1].set_ylabel('Expected Return (%)')
axes[1].set_title('Efficient Frontier\n(coloured = optimal for some lambda)', fontsize=11)
axes[1].legend(handles=legend_patches, fontsize=8)
axes[1].grid(alpha=0.3)

plt.suptitle('Risk-Aversion Sensitivity: How lambda Shifts the Optimal Portfolio', fontsize=13)
plt.tight_layout()
plt.show()